# March Machine Learning Mania 2026 — EDA

Table-by-table exploration of the raw competition data before any joining happens. Goals:

- confirm the actual file list / column names against the known March Mania schema
- row counts, dtypes, missingness per table
- season coverage per table (men's detailed stats only start 2003; women's tourney data starts later than men's)
- TeamID range/consistency checks (men 1000-1999, women 3000-3999)
- seed format parsing
- confirm season 2026 tourney results exist (our local "ground truth" to score against, since the competition already ran)


In [1]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

DATA_DIR = Path("../data")


In [2]:
files = sorted(DATA_DIR.glob("*.csv"))
print(f"{len(files)} CSV files found in {DATA_DIR.resolve()}\n")
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f"{f.name:45s} {size_kb:10,.1f} KB")


35 CSV files found in /sessions/confident-laughing-allen/mnt/kaggle-projects/march-machine-learning-mania-2026/data

Cities.csv                                           9.6 KB
Conferences.csv                                      1.6 KB
MConferenceTourneyGames.csv                        182.5 KB
MGameCities.csv                                  2,910.5 KB
MMasseyOrdinals.csv                            126,106.6 KB
MNCAATourneyCompactResults.csv                      75.9 KB
MNCAATourneyDetailedResults.csv                    141.5 KB
MNCAATourneySeedRoundSlots.csv                      15.5 KB
MNCAATourneySeeds.csv                               39.6 KB
MNCAATourneySlots.csv                               51.8 KB
MRegularSeasonCompactResults.csv                 5,683.9 KB
MRegularSeasonDetailedResults.csv               12,097.9 KB
MSeasons.csv                                         1.8 KB
MSecondaryTourneyCompactResults.csv                 62.1 KB
MSecondaryTourneyTeams.csv                 

In [3]:
def profile(df, name, id_cols=None):
    """Quick shape / dtype / missingness / head summary for one table."""
    print(f"=== {name} ===")
    print(f"shape: {df.shape}")
    print("\ndtypes:")
    print(df.dtypes)
    miss = df.isna().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    if len(miss):
        print("\nmissing %% (columns with any missing):")
        print((miss * 100).round(2))
    else:
        print("\nno missing values")
    if "Season" in df.columns:
        print(f"\nseason range: {df['Season'].min()} - {df['Season'].max()}, "
              f"{df['Season'].nunique()} unique seasons")
    print("\nhead:")
    display(df.head(3))
    print()
    return df


def load(pattern):
    matches = sorted(DATA_DIR.glob(pattern))
    if not matches:
        print(f"[missing] no file matches pattern: {pattern}")
        return None
    if len(matches) > 1:
        print(f"[note] multiple matches for {pattern}: {[m.name for m in matches]}, using first")
    return pd.read_csv(matches[0])


## Teams

TeamID ↔ name lookup. Men's IDs should be 1000-1999, women's 3000-3999.

In [4]:
m_teams = load("MTeams.csv")
w_teams = load("WTeams.csv")
if m_teams is not None:
    profile(m_teams, "MTeams")
    print("MTeamID range:", m_teams["TeamID"].min(), "-", m_teams["TeamID"].max())
if w_teams is not None:
    profile(w_teams, "WTeams")
    print("WTeamID range:", w_teams["TeamID"].min(), "-", w_teams["TeamID"].max())


=== MTeams ===
shape: (381, 4)

dtypes:
TeamID            int64
TeamName         object
FirstD1Season     int64
LastD1Season      int64
dtype: object

no missing values

head:
   TeamID     TeamName  FirstD1Season  LastD1Season
0    1101  Abilene Chr           2014          2026
1    1102    Air Force           1985          2026
2    1103        Akron           1985          2026

MTeamID range: 1101 - 1481
=== WTeams ===
shape: (379, 2)

dtypes:
TeamID       int64
TeamName    object
dtype: object

no missing values

head:
   TeamID     TeamName
0    3101  Abilene Chr
1    3102    Air Force
2    3103        Akron

WTeamID range: 3101 - 3481


## Seasons

Season metadata — day-zero date, region names. Defines which seasons exist per gender.

In [5]:
m_seasons = load("MSeasons.csv")
w_seasons = load("WSeasons.csv")
if m_seasons is not None:
    profile(m_seasons, "MSeasons")
if w_seasons is not None:
    profile(w_seasons, "WSeasons")


=== MSeasons ===
shape: (42, 6)

dtypes:
Season      int64
DayZero    object
RegionW    object
RegionX    object
RegionY    object
RegionZ    object
dtype: object

no missing values

season range: 1985 - 2026, 42 unique seasons

head:
   Season     DayZero RegionW    RegionX    RegionY    RegionZ
0    1985  10/29/1984    East       West    Midwest  Southeast
1    1986  10/28/1985    East    Midwest  Southeast       West
2    1987  10/27/1986    East  Southeast    Midwest       West

=== WSeasons ===
shape: (29, 6)

dtypes:
Season      int64
DayZero    object
RegionW    object
RegionX    object
RegionY    object
RegionZ    object
dtype: object

no missing values

season range: 1998 - 2026, 29 unique seasons

head:
   Season     DayZero RegionW  RegionX  RegionY RegionZ
0    1998  10/27/1997    East  Midwest  Mideast    West
1    1999  10/26/1998    East  Mideast  Midwest    West
2    2000  11/01/1999    East  Midwest  Mideast    West



## Tournament seeds

One row per team per season with its seed (e.g. `W01`, `X16a` for play-in teams). Check the format and how many teams per season (64-68ish).

In [6]:
m_seeds = load("MNCAATourneySeeds.csv")
w_seeds = load("WNCAATourneySeeds.csv")
if m_seeds is not None:
    profile(m_seeds, "MNCAATourneySeeds")
    print("sample seed values:", m_seeds["Seed"].sample(10, random_state=0).tolist())
    print("teams per season (men):")
    print(m_seeds.groupby("Season").size().tail(10))
if w_seeds is not None:
    profile(w_seeds, "WNCAATourneySeeds")
    print("teams per season (women):")
    print(w_seeds.groupby("Season").size().tail(10))


=== MNCAATourneySeeds ===
shape: (2694, 3)

dtypes:
Season     int64
Seed      object
TeamID     int64
dtype: object

no missing values

season range: 1985 - 2026, 41 unique seasons

head:
   Season Seed  TeamID
0    1985  W01    1207
1    1985  W02    1210
2    1985  W03    1228

sample seed values: ['X11', 'Z16a', 'Y05', 'Y08', 'Y01', 'X13', 'Y12', 'X01', 'W05', 'W03']
teams per season (men):
Season
2016    68
2017    68
2018    68
2019    68
2021    68
2022    68
2023    68
2024    68
2025    68
2026    68
dtype: int64
=== WNCAATourneySeeds ===
shape: (1812, 3)

dtypes:
Season     int64
Seed      object
TeamID     int64
dtype: object

no missing values

season range: 1998 - 2026, 28 unique seasons

head:
   Season Seed  TeamID
0    1998  W01    3330
1    1998  W02    3163
2    1998  W03    3112

teams per season (women):
Season
2016    64
2017    64
2018    64
2019    64
2021    64
2022    68
2023    68
2024    68
2025    68
2026    68
dtype: int64


## Regular season results (compact)

One row per regular-season game: winning/losing team, score, location, overtime periods.

In [7]:
m_rs_compact = load("MRegularSeasonCompactResults.csv")
w_rs_compact = load("WRegularSeasonCompactResults.csv")
if m_rs_compact is not None:
    profile(m_rs_compact, "MRegularSeasonCompactResults")
    print("games per season (men, last 10):")
    print(m_rs_compact.groupby("Season").size().tail(10))
    print("\nscore margin (WScore-LScore) describe:")
    print((m_rs_compact["WScore"] - m_rs_compact["LScore"]).describe())
    if "WLoc" in m_rs_compact.columns:
        print("\nWLoc value counts:")
        print(m_rs_compact["WLoc"].value_counts())
if w_rs_compact is not None:
    profile(w_rs_compact, "WRegularSeasonCompactResults")


=== MRegularSeasonCompactResults ===
shape: (198577, 8)

dtypes:
Season      int64
DayNum      int64
WTeamID     int64
WScore      int64
LTeamID     int64
LScore      int64
WLoc       object
NumOT       int64
dtype: object

no missing values

season range: 1985 - 2026, 42 unique seasons

head:
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    1985      20     1228      81     1328      64    N      0
1    1985      25     1106      77     1354      70    H      0
2    1985      25     1112      63     1223      56    H      0

games per season (men, last 10):
Season
2017    5395
2018    5405
2019    5463
2020    5328
2021    3855
2022    5345
2023    5602
2024    5607
2025    5641
2026    5647
dtype: int64

score margin (WScore-LScore) describe:
count    198577.000000
mean         12.087563
std           9.412255
min           1.000000
25%           5.000000
50%          10.000000
75%          17.000000
max          94.000000
dtype: float64

WLoc value counts:
WLoc
H

## Regular season results (detailed box scores)

Same games as compact, plus FG/3P/FT/rebound/turnover box score stats. Historically men's only from 2003 onward — check whether that gap still holds and whether women's detailed data has a similar cutoff.

In [8]:
m_rs_detail = load("MRegularSeasonDetailedResults.csv")
w_rs_detail = load("WRegularSeasonDetailedResults.csv")
if m_rs_detail is not None:
    profile(m_rs_detail, "MRegularSeasonDetailedResults")
    if m_rs_compact is not None:
        missing_seasons = sorted(set(m_rs_compact["Season"]) - set(m_rs_detail["Season"]))
        print(f"seasons in compact but not detailed (men): {missing_seasons}")
if w_rs_detail is not None:
    profile(w_rs_detail, "WRegularSeasonDetailedResults")
    if w_rs_compact is not None:
        missing_seasons = sorted(set(w_rs_compact["Season"]) - set(w_rs_detail["Season"]))
        print(f"seasons in compact but not detailed (women): {missing_seasons}")


=== MRegularSeasonDetailedResults ===
shape: (124529, 34)

dtypes:
Season      int64
DayNum      int64
WTeamID     int64
WScore      int64
LTeamID     int64
LScore      int64
WLoc       object
NumOT       int64
WFGM        int64
WFGA        int64
WFGM3       int64
WFGA3       int64
WFTM        int64
WFTA        int64
WOR         int64
WDR         int64
WAst        int64
WTO         int64
WStl        int64
WBlk        int64
WPF         int64
LFGM        int64
LFGA        int64
LFGM3       int64
LFGA3       int64
LFTM        int64
LFTA        int64
LOR         int64
LDR         int64
LAst        int64
LTO         int64
LStl        int64
LBlk        int64
LPF         int64
dtype: object

no missing values

season range: 2003 - 2026, 24 unique seasons

head:
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT  WFGM  WFGA  WFGM3  WFGA3  WFTM  WFTA  WOR  WDR  WAst  WTO  WStl  WBlk  WPF  LFGM  LFGA  LFGM3  LFGA3  LFTM  LFTA  LOR  LDR  LAst  LTO  LStl  LBlk  LPF
0    2003      10  

## Tournament results

The actual bracket outcomes. **This is what lets us score locally against 2026** — the competition already happened, so `Season == 2026` rows here are our ground truth.

In [9]:
m_tourney_compact = load("MNCAATourneyCompactResults.csv")
w_tourney_compact = load("WNCAATourneyCompactResults.csv")
if m_tourney_compact is not None:
    profile(m_tourney_compact, "MNCAATourneyCompactResults")
    has_2026 = (m_tourney_compact["Season"] == 2026).sum()
    print(f"men's 2026 tourney games present: {has_2026}")
    if has_2026:
        display(m_tourney_compact[m_tourney_compact["Season"] == 2026].head(10))
if w_tourney_compact is not None:
    profile(w_tourney_compact, "WNCAATourneyCompactResults")
    has_2026 = (w_tourney_compact["Season"] == 2026).sum()
    print(f"women's 2026 tourney games present: {has_2026}")


=== MNCAATourneyCompactResults ===
shape: (2585, 8)

dtypes:
Season      int64
DayNum      int64
WTeamID     int64
WScore      int64
LTeamID     int64
LScore      int64
WLoc       object
NumOT       int64
dtype: object

no missing values

season range: 1985 - 2025, 40 unique seasons

head:
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    1985     136     1116      63     1234      54    N      0
1    1985     136     1120      59     1345      58    N      0
2    1985     136     1207      68     1250      43    N      0

men's 2026 tourney games present: 0
=== WNCAATourneyCompactResults ===
shape: (1717, 8)

dtypes:
Season      int64
DayNum      int64
WTeamID     int64
WScore      int64
LTeamID     int64
LScore      int64
WLoc       object
NumOT       int64
dtype: object

no missing values

season range: 1998 - 2025, 27 unique seasons

head:
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    1998     137     3104      94     3422      46    H     

## Massey Ordinals

Third-party power-ranking systems over time (men's only, historically). Check which systems exist, coverage per season, and how close to end-of-season the rankings run.

In [10]:
m_ordinals = load("MMasseyOrdinals.csv")
if m_ordinals is not None:
    profile(m_ordinals, "MMasseyOrdinals")
    print("number of distinct ranking systems:", m_ordinals["SystemName"].nunique())
    print("\nsystems present in 2026 season:")
    print(m_ordinals.loc[m_ordinals["Season"] == 2026, "SystemName"].value_counts().head(20))
    print("\nmax RankingDayNum by season (last 10 seasons) — how late in the season rankings go:")
    print(m_ordinals.groupby("Season")["RankingDayNum"].max().tail(10))


=== MMasseyOrdinals ===
shape: (5865001, 5)

dtypes:
Season            int64
RankingDayNum     int64
SystemName       object
TeamID            int64
OrdinalRank       int64
dtype: object

no missing values

season range: 2003 - 2026, 24 unique seasons

head:
   Season  RankingDayNum SystemName  TeamID  OrdinalRank
0    2003             35        SEL    1102          159
1    2003             35        SEL    1103          229
2    2003             35        SEL    1104           12

number of distinct ranking systems: 197

systems present in 2026 season:
SystemName
PGH    7300
DII    7300
DOK    7300
WIL    7300
TRK    7300
POM    7300
KPK    7300
MAS    7300
MOR    7300
NOL    7298
TRP    6935
BWE    6935
WEI    6935
STY    6935
JNG    6935
HAS    6935
EMK    6935
INC    6935
EBP    6570
RMS    6570
Name: count, dtype: int64

max RankingDayNum by season (last 10 seasons) — how late in the season rankings go:
Season
2017    133
2018    133
2019    133
2020    128
2021    133
2022    13

## Conferences & team spellings

Conference membership by season, and the name-variant → TeamID lookup (useful for joining any external data later).

In [11]:
conferences = load("Conferences.csv")
m_team_conf = load("MTeamConferences.csv")
w_team_conf = load("WTeamConferences.csv")
m_spellings = load("MTeamSpellings.csv")
w_spellings = load("WTeamSpellings.csv")
for name, df in [("Conferences", conferences), ("MTeamConferences", m_team_conf),
                  ("WTeamConferences", w_team_conf), ("MTeamSpellings", m_spellings),
                  ("WTeamSpellings", w_spellings)]:
    if df is not None:
        profile(df, name)


=== Conferences ===
shape: (51, 2)

dtypes:
ConfAbbrev     object
Description    object
dtype: object

no missing values

head:
  ConfAbbrev                   Description
0      a_sun       Atlantic Sun Conference
1      a_ten        Atlantic 10 Conference
2        aac  American Athletic Conference

=== MTeamConferences ===
shape: (13753, 3)

dtypes:
Season         int64
TeamID         int64
ConfAbbrev    object
dtype: object

no missing values

season range: 1985 - 2026, 42 unique seasons

head:
   Season  TeamID ConfAbbrev
0    1985    1102        wac
1    1985    1103        ovc
2    1985    1104        sec

=== WTeamConferences ===
shape: (9853, 3)

dtypes:
Season         int64
TeamID         int64
ConfAbbrev    object
dtype: object

no missing values

season range: 1998 - 2026, 29 unique seasons

head:
   Season  TeamID ConfAbbrev
0    1998    3102        wac
1    1998    3103        mac
2    1998    3104        sec

=== MTeamSpellings ===
shape: (1178, 2)

dtypes:
TeamNameSpellin

## Sample submission

Confirms the exact format we need to produce: `ID = 2026_TeamIdLow_TeamIdHigh`, `Pred` = P(low beats high).

In [12]:
sub_files = sorted(DATA_DIR.glob("*ampleSubmission*.csv"))
print("sample submission files found:", [f.name for f in sub_files])
if sub_files:
    sample_sub = pd.read_csv(sub_files[0])
    profile(sample_sub, sub_files[0].name)
    ids = sample_sub["ID"].str.split("_", expand=True)
    ids.columns = ["Season", "TeamIdLow", "TeamIdHigh"]
    print("seasons referenced in submission ID:", ids["Season"].unique())
    print("row count:", len(sample_sub))


sample submission files found: ['SampleSubmissionStage1.csv', 'SampleSubmissionStage2.csv']
=== SampleSubmissionStage1.csv ===
shape: (519144, 2)

dtypes:
ID       object
Pred    float64
dtype: object

no missing values

head:
               ID  Pred
0  2022_1101_1102   0.5
1  2022_1101_1103   0.5
2  2022_1101_1104   0.5

seasons referenced in submission ID: ['2022' '2023' '2024' '2025']
row count: 519144


## Cross-table consistency checks

Do all TeamIDs referenced in results/seeds actually exist in the Teams table? Any seasons present in one table but missing from another?

In [13]:
if m_teams is not None and m_tourney_compact is not None:
    known_ids = set(m_teams["TeamID"])
    referenced_ids = set(m_tourney_compact["WTeamID"]) | set(m_tourney_compact["LTeamID"])
    unknown = referenced_ids - known_ids
    print(f"men's tourney TeamIDs not found in MTeams: {unknown if unknown else 'none'}")

if m_seeds is not None and m_tourney_compact is not None:
    seed_seasons = set(m_seeds["Season"])
    tourney_seasons = set(m_tourney_compact["Season"])
    print(f"seasons in seeds but not tourney results: {sorted(seed_seasons - tourney_seasons)}")
    print(f"seasons in tourney results but not seeds: {sorted(tourney_seasons - seed_seasons)}")


men's tourney TeamIDs not found in MTeams: none
seasons in seeds but not tourney results: [2026]
seasons in tourney results but not seeds: []


## Summary of findings

_Fill in after running against the real data:_

- Confirmed file list vs. expected schema:
- Missingness worth handling:
- Earliest usable season for detailed box-score features:
- Seed format quirks (play-in games, etc.):
- Row count in `MMasseyOrdinals` / systems available close to selection Sunday:
- Anything inconsistent across tables (ID mismatches, season gaps):
- Confirmed 2026 ground-truth results available for local scoring: yes/no
